In [23]:
import numpy as np
import pandas as pd

In [24]:
votes= pd.read_csv('https://voteview.com/static/data/out/votes/HS119_votes.csv')

In [25]:
votes.head(3).T

,0,1,2
congress,119,119,119
chamber,House,House,House
rollnumber,1,1,1
icpsr,14854,14863,14873
cast_code,1,1,6
prob,99.6,78.9,100.0


# pandas skills: filter columns

In [26]:
votes = votes[['chamber', 'rollnumber', 'icpsr', 'cast_code']]  # keep columns
# icpsr is a political research thing at U of Michigan
# or do this:
# votes = votes.drop(['congress', 'prob'], axis=1)  # drop columns
votes  # choose keep or drop but it's not necessary to have both

,chamber,rollnumber,icpsr,cast_code
0,House,1,14854,1
1,House,1,14863,1
2,House,1,14873,6
3,House,1,15029,6
4,House,1,15433,6
...,...,...,...,...
383026,Senate,915,42506,1
383027,Senate,915,42507,1
383028,Senate,915,42508,1
383029,Senate,915,49308,6


In [27]:
data1 = [{'id': 1, 'x': 'a'}, {'id': 1, 'x': 'b'}, {'id': 1, 'x': 'c'}]
data2 = [{'id': 1, 'y': 'd'}, {'id': 1, 'y': 'e'}, {'id': 1, 'y': 'f'}]

In [28]:
mergedata = pd.merge(pd.DataFrame(data1), pd.DataFrame(data2), on = 'id')
mergedata

,id,x,y
0,1,a,d
1,1,a,e
2,1,a,f
3,1,b,d
4,1,b,e
5,1,b,f
6,1,c,d
7,1,c,e
8,1,c,f


# pandas skill: many-to-many joins (usually we don't want this! but here we do)

In [29]:
vote_compare = pd.merge(votes, votes, on = ['chamber', 'rollnumber'])
vote_compare

,chamber,rollnumber,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,House,1,14854,1,14854,1
1,House,1,14854,1,14863,1
2,House,1,14854,1,14873,6
3,House,1,14854,1,15029,6
4,House,1,14854,1,15433,6
...,...,...,...,...,...,...
135162640,Senate,915,49703,1,42506,1
135162641,Senate,915,49703,1,42507,1
135162642,Senate,915,49703,1,42508,1
135162643,Senate,915,49703,1,49308,6


# pandas skill: more dropping

In [30]:
vote_compare = vote_compare.drop(['chamber', 'rollnumber'], axis=1)
vote_compare

,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,14854,1,14854,1
1,14854,1,14863,1
2,14854,1,14873,6
3,14854,1,15029,6
4,14854,1,15433,6
...,...,...,...,...
135162640,49703,1,42506,1
135162641,49703,1,42507,1
135162642,49703,1,42508,1
135162643,49703,1,49308,6


# pandas skill: create new column

In [31]:
vote_compare['agree'] = vote_compare['cast_code_x'] == vote_compare['cast_code_y']  # comparitive equals ==
vote_compare = vote_compare.drop(['cast_code_x', 'cast_code_y'], axis=1)
vote_compare

,icpsr_x,icpsr_y,agree
0,14854,14854,True
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
...,...,...,...
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True
135162643,49703,49308,False


# pandas skill: filter rows

In [32]:
vote_compare = vote_compare.query('icpsr_x != icpsr_y')
vote_compare

,icpsr_x,icpsr_y,agree
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
5,14854,15448,False
...,...,...,...
135162639,49703,42505,True
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True


# pandas skill: rename columns

In [33]:
vote_compare = vote_compare.rename({'icpsr_x': 'icpsr', 'icpsr_y': 'comparison_member'}, axis=1)
vote_compare  # comparing two people

,icpsr,comparison_member,agree
1,14854,14863,True
2,14854,14873,False
3,14854,15029,False
4,14854,15433,False
5,14854,15448,False
...,...,...,...
135162639,49703,42505,True
135162640,49703,42506,True
135162641,49703,42507,True
135162642,49703,42508,True


# pandas skill: sorting

In [34]:
vote_compare = vote_compare.sort_values(['icpsr', 'comparison_member'], ascending=True)
vote_compare

,icpsr,comparison_member,agree
125985887,14226,14435,False
125995688,14226,14435,False
126005292,14226,14435,False
126015093,14226,14435,False
126024894,14226,14435,False
...,...,...,...
37045188,99912,91980,True
37232677,99912,91980,True
37420166,99912,91980,True
37794279,99912,91980,True


# pandas skill: data aggregation

In [ ]:
vote_compare = vote_compare.groupby(['icpsr', 'comparison_member']).agg({'agree': 'mean'}).reset_index()  # had to reset row indices
vote_compare

,icpsr,comparison_member,agree
0,14226,14435,0.028415
1,14226,14858,0.077596
2,14226,14871,0.060109
3,14226,14921,0.798907
4,14226,15021,0.123497
...,...,...,...
215349,99912,42505,1.000000
215350,99912,42506,1.000000
215351,99912,49308,0.000000
215352,99912,49703,0.972414


In [38]:
# user supplies 14226, for example
user = '14226'
vote_compare.query(f'icpsr == {user}').sort_values('agree', ascending=True).head(10)

,icpsr,comparison_member,agree
0,14226,14435,0.028415
64,14226,41301,0.030601
37,14226,29147,0.031694
10,14226,20707,0.037158
8,14226,20330,0.046995
52,14226,40908,0.048087
2,14226,14871,0.060109
101,14226,49308,0.061202
29,14226,21706,0.062295
76,14226,41706,0.063388


In [41]:
import os
os.chdir('..')


from contrans import contrans
ct = contrans()


In [44]:
ct.get_vote_similarity_data()

KeyboardInterrupt: 

In [ ]:
vote_compare = pd.read_parquet('data/raw/vote_compare.parquet')
vote_compare